In [0]:
# COMMAND ----------
# ============================================================
# 09_ML_FEATURES_INFERENCE
# AML TRANSACTION MONITORING
# ============================================================
#
# PURPOSE
# -------
# 1. Read the complete Silver transaction history as context
# 2. Identify only new transactions for scoring
# 3. Recreate EXACTLY the same 28 ML features used in training
# 4. Do NOT retrain the model
# 5. Load the already registered model from Unity Catalog
# 6. Score only new transactions
# 7. Save ML features and predictions to S3-backed Delta tables
#
# ============================================================


from pyspark.sql import functions as F
from pyspark.sql.window import Window

import mlflow
import pandas as pd
import numpy as np


# COMMAND ----------
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

CATALOG = "aml_engine"
SCHEMA = "aml_poc"

SILVER_TX_TABLE = (
    f"{CATALOG}.{SCHEMA}.silver_transactions"
)

SILVER_ACCOUNTS_TABLE = (
    f"{CATALOG}.{SCHEMA}.silver_accounts"
)

GRAPH_RESULTS_TABLE = (
    f"{CATALOG}.{SCHEMA}.graph_results"
)

ML_FEATURES_TABLE = (
    f"{CATALOG}.{SCHEMA}.ml_features_inference"
)

ML_PREDICTIONS_TABLE = (
    f"{CATALOG}.{SCHEMA}.ml_predictions"
)

ML_SCORING_STATE_TABLE = (
    f"{CATALOG}.{SCHEMA}.ml_scoring_state"
)


# COMMAND ----------
# ============================================================
# S3 CONFIGURATION
# ============================================================

S3_BASE_PATH = (
    "s3://zubair-s3-demo/raw_dataset/aml"
)

S3_DELTA_PATH = (
    f"{S3_BASE_PATH}/delta_tables"
)

ML_FEATURES_PATH = (
    f"{S3_DELTA_PATH}/ml_features_inference"
)

ML_PREDICTIONS_PATH = (
    f"{S3_DELTA_PATH}/ml_predictions"
)

ML_SCORING_STATE_PATH = (
    f"{S3_DELTA_PATH}/ml_scoring_state"
)


# COMMAND ----------
# ============================================================
# MODEL CONFIGURATION
# ============================================================
#
# Replace YOUR_REGISTERED_MODEL with the actual model name
# registered inside:
#
# aml_engine.aml_poc
#
# Example:
# models:/aml_engine.aml_poc.aml_xgboost_model@Champion
#
# ============================================================

MODEL_URI = (
    "models:/aml_engine.aml_poc.YOUR_REGISTERED_MODEL@Champion"
)

MODEL_NAME = "aml_xgboost_final"

if "YOUR_REGISTERED_MODEL" in MODEL_URI:
    raise ValueError(
        "Set MODEL_URI and MODEL_NAME to your actual Unity Catalog registered model before running the Job."
    )


# COMMAND ----------
# ============================================================
# INCREMENTAL PROCESSING CONFIGURATION
# ============================================================
#
# IMPORTANT:
# _silver_processed_timestamp must be present in your Silver
# transaction table.
#
# This timestamp represents when a transaction reached Silver.
#
# DO NOT use IS_FRAUD to identify new records.
#
# Historical rows can also potentially have NULL labels.
#
# ============================================================

SILVER_PROCESS_TS_COL = "_silver_processed_timestamp"


# ============================================================
# INITIAL SCORING CUTOFF
# ============================================================
#
# Set this ONCE before the first production scoring run.
#
# Example:
#
# SCORING_START_TIME = "2026-09-16 14:00:00"
#
# Everything arriving after this timestamp can be considered
# production scoring data.
#
# ============================================================

SCORING_START_TIME = "2026-09-16 14:00:00"


# COMMAND ----------
# ============================================================
# FEATURE PARAMETERS
# ============================================================

LOOKBACK_WINDOW = 5

HIGH_VALUE_THRESHOLD = 1000.0

VELOCITY_THRESHOLD = 5

FAN_IN_THRESHOLD = 5

FAN_OUT_THRESHOLD = 5


# COMMAND ----------
# ============================================================
# MODEL FEATURE LIST
# ============================================================
#
# EXACT 28 FEATURES USED BY YOUR TRAINING NOTEBOOK
#
# ============================================================

FEATURE_COLUMNS = [

    "tx_amount",
    "tx_type",
    "event_time",

    "sender_country",
    "receiver_country",

    "sender_account_type",
    "receiver_account_type",

    "sender_init_balance",
    "receiver_init_balance",

    "sender_tx_count_before",
    "receiver_tx_count_before",

    "sender_total_amount_before",
    "receiver_total_amount_before",

    "sender_velocity_count",
    "receiver_velocity_count",

    "unique_receivers_before",
    "unique_senders_before",

    "high_value_flag",
    "velocity_flag",

    "fan_in_flag",
    "fan_out_flag",

    "fan_in_feature",
    "fan_out_feature",

    "sender_out_degree_before",
    "receiver_in_degree_before",

    "cycle_detected_as_of_time",
    "cycle_count"
]

print("Number of model features:", len(FEATURE_COLUMNS))


# COMMAND ----------
# ============================================================
# VALIDATE SILVER TABLE
# ============================================================

silver_columns = spark.table(
    SILVER_TX_TABLE
).columns

print("Silver columns:")
print(silver_columns)


# COMMAND ----------
# ============================================================
# CHECK REQUIRED PROCESSING TIMESTAMP
# ============================================================

if SILVER_PROCESS_TS_COL not in silver_columns:

    raise Exception(
        f"""
Required column '{SILVER_PROCESS_TS_COL}' was not found
in {SILVER_TX_TABLE}.

The incremental scoring logic requires a Silver processing
timestamp.

Available columns:
{silver_columns}
"""
    )


# COMMAND ----------
# ============================================================
# READ COMPLETE SILVER TRANSACTION HISTORY
# ============================================================
#
# IMPORTANT
# ----------
# We intentionally read ALL Silver transactions here.
#
# Why?
#
# New transaction features depend on historical behaviour:
#
# sender_tx_count_before
# receiver_tx_count_before
# velocity
# fan-in/fan-out
# graph degree
# cycle information
#
# We do NOT score all these rows.
#
# Later we select only NEW transactions.
#
# ============================================================

transactions = (
    spark.table(SILVER_TX_TABLE)
    .select(
        "tx_id",
        "sender_account_id",
        "receiver_account_id",
        "tx_type",
        "tx_amount",
        "event_time",
        SILVER_PROCESS_TS_COL
    )
)

print(
    "Total Silver transactions:",
    transactions.count()
)


# COMMAND ----------
# ============================================================
# IDENTIFY NEW TRANSACTIONS
# ============================================================
#
# Only records after SCORING_START_TIME are eligible.
#
# Then we perform an anti-join against the prediction table.
#
# This gives an additional protection against scoring the same
# transaction more than once.
#
# ============================================================

new_transactions = (
    transactions
    .filter(
        F.col(SILVER_PROCESS_TS_COL)
        >= F.to_timestamp(
            F.lit(SCORING_START_TIME)
        )
    )
    .select(
        "tx_id",
        SILVER_PROCESS_TS_COL
    )
)


# COMMAND ----------
# ============================================================
# CHECK WHETHER PREDICTION TABLE ALREADY EXISTS
# ============================================================

prediction_table_exists = (
    spark.catalog.tableExists(
        ML_PREDICTIONS_TABLE
    )
)

print(
    "Prediction table exists:",
    prediction_table_exists
)


# COMMAND ----------
# ============================================================
# REMOVE ALREADY-SCORED TRANSACTIONS
# ============================================================

if prediction_table_exists:

    scored_tx_ids = (
        spark.table(
            ML_PREDICTIONS_TABLE
        )
        .select("tx_id")
        .dropDuplicates()
    )

    new_transactions = (
        new_transactions
        .join(
            scored_tx_ids,
            on="tx_id",
            how="left_anti"
        )
    )


# COMMAND ----------
# ============================================================
# COUNT NEW TRANSACTIONS
# ============================================================

new_transaction_count = (
    new_transactions.count()
)

print(
    "Transactions waiting for ML scoring:",
    new_transaction_count
)


# COMMAND ----------
# ============================================================
# EXIT CLEANLY WHEN NOTHING IS NEW
# ============================================================

if new_transaction_count == 0:

    print(
        "No new transactions found. "
        "ML scoring is not required."
    )

else:

    print(
        f"Processing {new_transaction_count} new transactions."
    )


# COMMAND ----------
# ============================================================
# CREATE ML WORKING DATASET
# ============================================================
#
# Only this dataset will eventually be emitted/scored.
#
# Historical Silver data remains available separately in
# 'transactions' for feature calculations.
#
# ============================================================

if new_transaction_count > 0:

    new_tx_ids = (
        new_transactions
        .select("tx_id")
        .dropDuplicates()
    )

    print(
        "New transaction IDs prepared."
    )


# COMMAND ----------
# ============================================================
# CREATE TRANSACTION DATASET FOR FEATURE ENGINEERING
# ============================================================

if new_transaction_count > 0:

    ml_df = (
        transactions
        .drop(SILVER_PROCESS_TS_COL)
    )

    print(
        "Historical context loaded for feature engineering."
    )


# COMMAND ----------
# ============================================================
# LOAD ACCOUNT MASTER DATA
# ============================================================

if new_transaction_count > 0:

    accounts = spark.table(
        SILVER_ACCOUNTS_TABLE
    )

    print(
        "Account count:",
        accounts.count()
    )


# COMMAND ----------
# ============================================================
# SENDER ACCOUNT FEATURES
# ============================================================

if new_transaction_count > 0:

    sender_accounts = (
        accounts
        .select(

            F.col("account_id")
                .alias("sender_account_id"),

            F.col("country")
                .alias("sender_country"),

            F.col("account_type")
                .alias("sender_account_type"),

            F.col("init_balance")
                .alias("sender_init_balance")
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER ACCOUNT FEATURES
# ============================================================

if new_transaction_count > 0:

    receiver_accounts = (
        accounts
        .select(

            F.col("account_id")
                .alias("receiver_account_id"),

            F.col("country")
                .alias("receiver_country"),

            F.col("account_type")
                .alias("receiver_account_type"),

            F.col("init_balance")
                .alias("receiver_init_balance")
        )
    )


# COMMAND ----------
# ============================================================
# JOIN ACCOUNT INFORMATION
# ============================================================

if new_transaction_count > 0:

    ml_df = (
        ml_df

        .join(
            sender_accounts,
            on="sender_account_id",
            how="left"
        )

        .join(
            receiver_accounts,
            on="receiver_account_id",
            how="left"
        )
    )


# COMMAND ----------
# ============================================================
# SENDER HISTORICAL WINDOW
# ============================================================

if new_transaction_count > 0:

    sender_history_window = (
        Window
        .partitionBy(
            "sender_account_id"
        )
        .orderBy(
            "event_time"
        )
        .rangeBetween(
            Window.unboundedPreceding,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# SENDER TRANSACTION HISTORY
# ============================================================

if new_transaction_count > 0:

    ml_df = (
        ml_df

        .withColumn(
            "sender_tx_count_before",

            F.count("tx_id")
            .over(
                sender_history_window
            )
        )

        .withColumn(
            "sender_total_amount_before",

            F.coalesce(

                F.sum("tx_amount")
                .over(
                    sender_history_window
                ),

                F.lit(0.0)
            )
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER HISTORICAL WINDOW
# ============================================================

if new_transaction_count > 0:

    receiver_history_window = (
        Window
        .partitionBy(
            "receiver_account_id"
        )
        .orderBy(
            "event_time"
        )
        .rangeBetween(
            Window.unboundedPreceding,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER TRANSACTION HISTORY
# ============================================================

if new_transaction_count > 0:

    ml_df = (
        ml_df

        .withColumn(
            "receiver_tx_count_before",

            F.count("tx_id")
            .over(
                receiver_history_window
            )
        )

        .withColumn(
            "receiver_total_amount_before",

            F.coalesce(

                F.sum("tx_amount")
                .over(
                    receiver_history_window
                ),

                F.lit(0.0)
            )
        )
    )


# COMMAND ----------
# ============================================================
# SENDER VELOCITY WINDOW
# ============================================================

if new_transaction_count > 0:

    sender_velocity_window = (
        Window
        .partitionBy(
            "sender_account_id"
        )
        .orderBy(
            "event_time"
        )
        .rangeBetween(
            -LOOKBACK_WINDOW,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# SENDER VELOCITY
# ============================================================

if new_transaction_count > 0:

    ml_df = (
        ml_df
        .withColumn(
            "sender_velocity_count",

            F.count("tx_id")
            .over(
                sender_velocity_window
            )
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER VELOCITY WINDOW
# ============================================================

if new_transaction_count > 0:

    receiver_velocity_window = (
        Window
        .partitionBy(
            "receiver_account_id"
        )
        .orderBy(
            "event_time"
        )
        .rangeBetween(
            -LOOKBACK_WINDOW,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER VELOCITY
# ============================================================

if new_transaction_count > 0:

    ml_df = (
        ml_df
        .withColumn(
            "receiver_velocity_count",

            F.count("tx_id")
            .over(
                receiver_velocity_window
            )
        )
    )


# COMMAND ----------
# ============================================================
# FIRST SENDER-RECEIVER RELATIONSHIP
# ============================================================

if new_transaction_count > 0:

    sender_receiver_first_seen = (

        transactions

        .groupBy(
            "sender_account_id",
            "receiver_account_id"
        )

        .agg(

            F.min("event_time")
            .alias("first_seen_time")
        )
    )


# COMMAND ----------
# ============================================================
# NEW RECEIVER COUNTS
# ============================================================

if new_transaction_count > 0:

    new_receiver_counts = (

        sender_receiver_first_seen

        .groupBy(
            "sender_account_id",
            "first_seen_time"
        )

        .agg(

            F.count("*")
            .alias("new_receivers")
        )
    )


# COMMAND ----------
# ============================================================
# UNIQUE RECEIVER HISTORY
# ============================================================

if new_transaction_count > 0:

    unique_receiver_history_window = (

        Window

        .partitionBy(
            "sender_account_id"
        )

        .orderBy(
            "first_seen_time"
        )

        .rangeBetween(
            Window.unboundedPreceding,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# UNIQUE RECEIVERS BEFORE CURRENT TRANSACTION
# ============================================================

if new_transaction_count > 0:

    new_receiver_counts = (

        new_receiver_counts

        .withColumn(

            "unique_receivers_before",

            F.coalesce(

                F.sum("new_receivers")
                .over(
                    unique_receiver_history_window
                ),

                F.lit(0)
            )
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER LOOKUP
# ============================================================

if new_transaction_count > 0:

    receiver_lookup = (

        new_receiver_counts

        .select(

            "sender_account_id",

            F.col("first_seen_time")
            .alias("relationship_time"),

            "unique_receivers_before"
        )
    )


# COMMAND ----------
# ============================================================
# JOIN UNIQUE RECEIVER HISTORY
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .join(

            receiver_lookup,

            (
                (ml_df.sender_account_id ==
                 receiver_lookup.sender_account_id)

                &

                (ml_df.event_time ==
                 receiver_lookup.relationship_time)
            ),

            "left"
        )

        .drop(
            receiver_lookup.sender_account_id,
            "relationship_time"
        )

        .fillna(
            {
                "unique_receivers_before": 0
            }
        )
    )


# COMMAND ----------
# ============================================================
# FIRST RECEIVER-SENDER RELATIONSHIP
# ============================================================

if new_transaction_count > 0:

    receiver_sender_first_seen = (

        transactions

        .groupBy(
            "receiver_account_id",
            "sender_account_id"
        )

        .agg(

            F.min("event_time")
            .alias("first_seen_time")
        )
    )


# COMMAND ----------
# ============================================================
# NEW SENDER COUNTS
# ============================================================

if new_transaction_count > 0:

    new_sender_counts = (

        receiver_sender_first_seen

        .groupBy(
            "receiver_account_id",
            "first_seen_time"
        )

        .agg(

            F.count("*")
            .alias("new_senders")
        )
    )


# COMMAND ----------
# ============================================================
# UNIQUE SENDER HISTORY WINDOW
# ============================================================

if new_transaction_count > 0:

    unique_sender_history_window = (

        Window

        .partitionBy(
            "receiver_account_id"
        )

        .orderBy(
            "first_seen_time"
        )

        .rangeBetween(
            Window.unboundedPreceding,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# UNIQUE SENDERS BEFORE CURRENT TRANSACTION
# ============================================================

if new_transaction_count > 0:

    new_sender_counts = (

        new_sender_counts

        .withColumn(

            "unique_senders_before",

            F.coalesce(

                F.sum("new_senders")
                .over(
                    unique_sender_history_window
                ),

                F.lit(0)
            )
        )
    )


# COMMAND ----------
# ============================================================
# SENDER LOOKUP
# ============================================================

if new_transaction_count > 0:

    sender_lookup = (

        new_sender_counts

        .select(

            "receiver_account_id",

            F.col("first_seen_time")
            .alias("relationship_time"),

            "unique_senders_before"
        )
    )


# COMMAND ----------
# ============================================================
# JOIN UNIQUE SENDER HISTORY
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .join(

            sender_lookup,

            (
                (ml_df.receiver_account_id ==
                 sender_lookup.receiver_account_id)

                &

                (ml_df.event_time ==
                 sender_lookup.relationship_time)
            ),

            "left"
        )

        .drop(
            sender_lookup.receiver_account_id,
            "relationship_time"
        )

        .fillna(
            {
                "unique_senders_before": 0
            }
        )
    )


# COMMAND ----------
# ============================================================
# HIGH VALUE FLAG
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .withColumn(

            "high_value_flag",

            F.when(

                F.col("tx_amount")
                > HIGH_VALUE_THRESHOLD,

                1

            ).otherwise(0)
        )
    )


# COMMAND ----------
# ============================================================
# VELOCITY FLAG
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .withColumn(

            "velocity_flag",

            F.when(

                F.col("sender_velocity_count")
                >= VELOCITY_THRESHOLD,

                1

            ).otherwise(0)
        )
    )


# COMMAND ----------
# ============================================================
# FAN-IN FEATURE
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .withColumn(

            "fan_in_feature",

            F.col("unique_senders_before")
        )
    )


# COMMAND ----------
# ============================================================
# FAN-IN FLAG
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .withColumn(

            "fan_in_flag",

            F.when(

                F.col("fan_in_feature")
                >= FAN_IN_THRESHOLD,

                1

            ).otherwise(0)
        )
    )


# COMMAND ----------
# ============================================================
# FAN-OUT FEATURE
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .withColumn(

            "fan_out_feature",

            F.col("unique_receivers_before")
        )
    )


# COMMAND ----------
# ============================================================
# FAN-OUT FLAG
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .withColumn(

            "fan_out_flag",

            F.when(

                F.col("fan_out_feature")
                >= FAN_OUT_THRESHOLD,

                1

            ).otherwise(0)
        )
    )


# COMMAND ----------
# ============================================================
# SENDER EDGE HISTORY
# ============================================================

if new_transaction_count > 0:

    sender_edges = (

        transactions

        .select(
            "sender_account_id",
            "receiver_account_id",
            "event_time"
        )

        .dropDuplicates()
    )


# COMMAND ----------
# ============================================================
# FIRST OUTGOING EDGE
# ============================================================

if new_transaction_count > 0:

    sender_edge_first_seen = (

        sender_edges

        .groupBy(
            "sender_account_id",
            "receiver_account_id"
        )

        .agg(

            F.min("event_time")
            .alias("first_seen_time")
        )
    )


# COMMAND ----------
# ============================================================
# NEW OUTGOING EDGES
# ============================================================

if new_transaction_count > 0:

    sender_new_edges = (

        sender_edge_first_seen

        .groupBy(
            "sender_account_id",
            "first_seen_time"
        )

        .agg(

            F.count("*")
            .alias("new_outgoing_edges")
        )
    )


# COMMAND ----------
# ============================================================
# SENDER DEGREE WINDOW
# ============================================================

if new_transaction_count > 0:

    sender_degree_window = (

        Window

        .partitionBy(
            "sender_account_id"
        )

        .orderBy(
            "first_seen_time"
        )

        .rangeBetween(
            Window.unboundedPreceding,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# SENDER OUT DEGREE BEFORE
# ============================================================

if new_transaction_count > 0:

    sender_new_edges = (

        sender_new_edges

        .withColumn(

            "sender_out_degree_before",

            F.coalesce(

                F.sum(
                    "new_outgoing_edges"
                )
                .over(
                    sender_degree_window
                ),

                F.lit(0)
            )
        )
    )


# COMMAND ----------
# ============================================================
# SENDER DEGREE LOOKUP
# ============================================================

if new_transaction_count > 0:

    sender_degree_lookup = (

        sender_new_edges

        .select(

            "sender_account_id",

            F.col("first_seen_time")
            .alias("degree_event_time"),

            "sender_out_degree_before"
        )
    )


# COMMAND ----------
# ============================================================
# JOIN SENDER DEGREE
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .join(

            sender_degree_lookup,

            (
                (ml_df.sender_account_id ==
                 sender_degree_lookup.sender_account_id)

                &

                (ml_df.event_time ==
                 sender_degree_lookup.degree_event_time)
            ),

            "left"
        )

        .drop(
            sender_degree_lookup.sender_account_id,
            "degree_event_time"
        )

        .fillna(
            {
                "sender_out_degree_before": 0
            }
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER EDGE HISTORY
# ============================================================

if new_transaction_count > 0:

    receiver_edges = (

        transactions

        .select(
            "receiver_account_id",
            "sender_account_id",
            "event_time"
        )

        .dropDuplicates()
    )


# COMMAND ----------
# ============================================================
# FIRST INCOMING EDGE
# ============================================================

if new_transaction_count > 0:

    receiver_edge_first_seen = (

        receiver_edges

        .groupBy(
            "receiver_account_id",
            "sender_account_id"
        )

        .agg(

            F.min("event_time")
            .alias("first_seen_time")
        )
    )


# COMMAND ----------
# ============================================================
# NEW INCOMING EDGES
# ============================================================

if new_transaction_count > 0:

    receiver_new_edges = (

        receiver_edge_first_seen

        .groupBy(
            "receiver_account_id",
            "first_seen_time"
        )

        .agg(

            F.count("*")
            .alias("new_incoming_edges")
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER DEGREE WINDOW
# ============================================================

if new_transaction_count > 0:

    receiver_degree_window = (

        Window

        .partitionBy(
            "receiver_account_id"
        )

        .orderBy(
            "first_seen_time"
        )

        .rangeBetween(
            Window.unboundedPreceding,
            -1
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER IN DEGREE BEFORE
# ============================================================

if new_transaction_count > 0:

    receiver_new_edges = (

        receiver_new_edges

        .withColumn(

            "receiver_in_degree_before",

            F.coalesce(

                F.sum(
                    "new_incoming_edges"
                )
                .over(
                    receiver_degree_window
                ),

                F.lit(0)
            )
        )
    )


# COMMAND ----------
# ============================================================
# RECEIVER DEGREE LOOKUP
# ============================================================

if new_transaction_count > 0:

    receiver_degree_lookup = (

        receiver_new_edges

        .select(

            "receiver_account_id",

            F.col("first_seen_time")
            .alias("degree_event_time"),

            "receiver_in_degree_before"
        )
    )


# COMMAND ----------
# ============================================================
# JOIN RECEIVER DEGREE
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .join(

            receiver_degree_lookup,

            (
                (ml_df.receiver_account_id ==
                 receiver_degree_lookup.receiver_account_id)

                &

                (ml_df.event_time ==
                 receiver_degree_lookup.degree_event_time)
            ),

            "left"
        )

        .drop(
            receiver_degree_lookup.receiver_account_id,
            "degree_event_time"
        )

        .fillna(
            {
                "receiver_in_degree_before": 0
            }
        )
    )


# COMMAND ----------
# ============================================================
# LOAD GRAPH RESULTS
# ============================================================

if new_transaction_count > 0:

    graph_results = spark.table(
        GRAPH_RESULTS_TABLE
    )

    print(
        "Graph results loaded."
    )


# COMMAND ----------
# ============================================================
# MAP GRAPH CYCLE TRANSACTIONS TO EVENT TIME
# ============================================================

if new_transaction_count > 0:

    cycle_times = (

        graph_results

        .select(
            "cycle_id",
            "tx_id"
        )

        .dropDuplicates()

        .join(

            transactions.select(
                "tx_id",
                "event_time"
            ),

            on="tx_id",

            how="inner"
        )
    )


# COMMAND ----------
# ============================================================
# CYCLE COMPLETION TIME
# ============================================================

if new_transaction_count > 0:

    cycle_completion = (

        cycle_times

        .groupBy(
            "cycle_id"
        )

        .agg(

            F.max("event_time")
            .alias("cycle_completion_time")
        )
    )


# COMMAND ----------
# ============================================================
# TRANSACTION-LEVEL CYCLE FEATURES
# ============================================================

if new_transaction_count > 0:

    cycle_transaction_features = (

        cycle_times

        .join(

            cycle_completion,

            on="cycle_id",

            how="left"
        )
    )


# COMMAND ----------
# ============================================================
# CYCLE DETECTION AS OF TRANSACTION TIME
# ============================================================

if new_transaction_count > 0:

    cycle_features = (

        cycle_transaction_features

        .groupBy(
            "tx_id"
        )

        .agg(

            F.max(

                F.when(

                    F.col("event_time")
                    >= F.col(
                        "cycle_completion_time"
                    ),

                    1

                ).otherwise(0)

            ).alias(
                "cycle_detected_as_of_time"
            ),

            F.countDistinct(
                "cycle_id"
            ).alias(
                "cycle_count"
            )
        )
    )


# COMMAND ----------
# ============================================================
# JOIN CYCLE FEATURES
# ============================================================

if new_transaction_count > 0:

    ml_df = (

        ml_df

        .join(
            cycle_features,
            on="tx_id",
            how="left"
        )

        .fillna(
            {
                "cycle_detected_as_of_time": 0,
                "cycle_count": 0
            }
        )
    )


# COMMAND ----------
# ============================================================
# FILL NUMERIC NULL VALUES
# ============================================================

if new_transaction_count > 0:

    numeric_features = [

        "sender_tx_count_before",
        "receiver_tx_count_before",

        "sender_total_amount_before",
        "receiver_total_amount_before",

        "sender_velocity_count",
        "receiver_velocity_count",

        "unique_receivers_before",
        "unique_senders_before",

        "fan_in_feature",
        "fan_out_feature",

        "high_value_flag",
        "velocity_flag",

        "fan_in_flag",
        "fan_out_flag",

        "sender_out_degree_before",
        "receiver_in_degree_before",

        "cycle_detected_as_of_time",
        "cycle_count"
    ]

    ml_df = (
        ml_df.fillna(
            0,
            subset=numeric_features
        )
    )


# COMMAND ----------
# ============================================================
# KEEP ONLY NEW TRANSACTIONS
# ============================================================
#
# IMPORTANT
# ----------
# All feature calculations above used COMPLETE Silver history.
#
# Now we restrict the output to only newly arrived transactions.
#
# ============================================================

if new_transaction_count > 0:

    ml_features_new = (

        ml_df

        .join(
            new_tx_ids,
            on="tx_id",
            how="inner"
        )
    )


# COMMAND ----------
# ============================================================
# SELECT EXACT MODEL FEATURES
# ============================================================

if new_transaction_count > 0:

    ml_features_new = (

        ml_features_new

        .select(
            "tx_id",
            *FEATURE_COLUMNS
        )
    )


# COMMAND ----------
# ============================================================
# VALIDATE FEATURE SCHEMA
# ============================================================

if new_transaction_count > 0:

    actual_features = [
        c
        for c in ml_features_new.columns
        if c != "tx_id"
    ]

    missing_features = [
        c
        for c in FEATURE_COLUMNS
        if c not in actual_features
    ]

    extra_features = [
        c
        for c in actual_features
        if c not in FEATURE_COLUMNS
    ]

    if missing_features:

        raise Exception(
            f"Missing model features: {missing_features}"
        )

    print(
        "Feature schema validation passed."
    )

    print(
        "Feature count:",
        len(actual_features)
    )


# COMMAND ----------
# ============================================================
# DATA QUALITY CHECK
# ============================================================

if new_transaction_count > 0:

    display(
        ml_features_new.limit(20)
    )


# COMMAND ----------
# ============================================================
# SAVE NEW ML FEATURES
# ============================================================
#
# Append only.
#
# We do NOT overwrite the ML feature table because this is
# production/incremental data.
#
# ============================================================

if new_transaction_count > 0:

    (
        ml_features_new

        .withColumn(
            "feature_generation_timestamp",
            F.current_timestamp()
        )

        .write

        .format("delta")

        .mode("append")

        .option(
            "path",
            ML_FEATURES_PATH
        )

        .saveAsTable(
            ML_FEATURES_TABLE
        )
    )

    print(
        "New ML features saved:",
        ML_FEATURES_TABLE
    )

    print(
        "S3 location:",
        ML_FEATURES_PATH
    )


# COMMAND ----------
# ============================================================
# LOAD REGISTERED MODEL
# ============================================================

if new_transaction_count > 0:

    print(
        "Loading registered model:"
    )

    print(
        MODEL_URI
    )

    model = mlflow.pyfunc.load_model(
        MODEL_URI
    )

    print(
        "Registered model loaded successfully."
    )


# COMMAND ----------
# ============================================================
# CONVERT ONLY NEW ROWS TO PANDAS
# ============================================================
#
# We are NOT converting the entire Silver table.
#
# Only the current new batch goes to Pandas.
#
# ============================================================

if new_transaction_count > 0:

    prediction_input_spark = (

        ml_features_new

        .select(
            "tx_id",
            *FEATURE_COLUMNS
        )
    )

    prediction_input_pdf = (

        prediction_input_spark

        .toPandas()
    )

    print(
        "Rows sent to model:",
        len(prediction_input_pdf)
    )


# COMMAND ----------
# ============================================================
# PREPARE MODEL INPUT
# ============================================================

if new_transaction_count > 0:

    X_new = (
        prediction_input_pdf[
            FEATURE_COLUMNS
        ]
    )

    print(
        "Model input columns:"
    )

    print(
        list(X_new.columns)
    )


# COMMAND ----------
# ============================================================
# MODEL PREDICTION
# ============================================================
#
# This assumes your registered model was logged so that
# model.predict() returns the model output.
#
# ============================================================

if new_transaction_count > 0:

    raw_predictions = (
        model.predict(X_new)
    )

    print(
        "Prediction completed."
    )


# COMMAND ----------
# ============================================================
# NORMALIZE MODEL OUTPUT
# ============================================================
#
# Different MLflow registrations can return:
#
# 1. one-dimensional predictions
# 2. a pandas DataFrame
# 3. probability columns
#
# This section handles the common outputs.
#
# ============================================================

if new_transaction_count > 0:

    if isinstance(
        raw_predictions,
        pd.DataFrame
    ):

        prediction_pdf = raw_predictions.copy()

    else:

        prediction_array = np.asarray(
            raw_predictions
        )

        if prediction_array.ndim == 1:

            prediction_pdf = pd.DataFrame(
                {
                    "prediction": prediction_array
                }
            )

        elif prediction_array.ndim == 2:

            if prediction_array.shape[1] == 1:

                prediction_pdf = pd.DataFrame(
                    {
                        "prediction":
                        prediction_array[:, 0]
                    }
                )

            else:

                prediction_pdf = pd.DataFrame(
                    {
                        "prediction_class_0":
                        prediction_array[:, 0],

                        "prediction_class_1":
                        prediction_array[:, 1]
                    }
                )

        else:

            raise Exception(
                "Unsupported model prediction output."
            )


# COMMAND ----------
# ============================================================
# CREATE ML SCORE
# ============================================================
#
# Preferred case:
# model output contains probability for fraud.
#
# If your model returns two columns, column 1 is treated as
# the probability of class 1.
#
# Otherwise prediction itself is used.
#
# ============================================================

if new_transaction_count > 0:

    if (
        "prediction_class_1"
        in prediction_pdf.columns
    ):

        prediction_pdf["ml_score"] = (

            prediction_pdf[
                "prediction_class_1"
            ]
            .astype(float)
        )

        prediction_pdf["prediction"] = (

            prediction_pdf[
                "prediction_class_1"
            ]
            >= 0.5
        ).astype(int)

    elif "prediction" in prediction_pdf.columns:

        prediction_pdf["ml_score"] = (

            prediction_pdf[
                "prediction"
            ]
            .astype(float)
        )

        prediction_pdf["prediction"] = (

            prediction_pdf[
                "prediction"
            ]
            .astype(float)
            >= 0.5
        ).astype(int)

    else:

        raise Exception(
            f"Could not determine prediction output columns: "
            f"{list(prediction_pdf.columns)}"
        )


# COMMAND ----------
# ============================================================
# BUILD PREDICTION DATAFRAME
# ============================================================

if new_transaction_count > 0:

    prediction_output_pdf = pd.DataFrame(
        {
            "tx_id":
                prediction_input_pdf[
                    "tx_id"
                ].values,

            "ml_score":
                prediction_pdf[
                    "ml_score"
                ].values,

            "prediction":
                prediction_pdf[
                    "prediction"
                ].astype(int).values,

            "model_name":
                MODEL_NAME,

            "model_uri":
                MODEL_URI,

            "prediction_timestamp":
                pd.Timestamp.now(tz="UTC").tz_localize(None)
        }
    )

    print(
        "Prediction output rows:",
        len(prediction_output_pdf)
    )


# COMMAND ----------
# ============================================================
# CONVERT PREDICTIONS TO SPARK
# ============================================================

if new_transaction_count > 0:

    predictions_spark = (

        spark.createDataFrame(
            prediction_output_pdf
        )
    )


# COMMAND ----------
# ============================================================
# SAVE ML PREDICTIONS
# ============================================================
#
# Append-only prediction history.
#
# ============================================================

if new_transaction_count > 0:

    (
        predictions_spark

        .write

        .format("delta")

        .mode("append")

        .option(
            "path",
            ML_PREDICTIONS_PATH
        )

        .saveAsTable(
            ML_PREDICTIONS_TABLE
        )
    )

    print(
        "Predictions saved:"
    )

    print(
        ML_PREDICTIONS_TABLE
    )

    print(
        "S3 location:"
    )

    print(
        ML_PREDICTIONS_PATH
    )


# COMMAND ----------
# ============================================================
# UPDATE SCORING STATE
# ============================================================
#
# The state records the newest Silver processing timestamp
# successfully handled by this run.
#
# ============================================================

if new_transaction_count > 0:

    current_max_processing_timestamp = (

        transactions

        .join(
            new_tx_ids,
            on="tx_id",
            how="inner"
        )

        .agg(
            F.max(
                SILVER_PROCESS_TS_COL
            )
            .alias(
                "last_processed_timestamp"
            )
        )
    )


# COMMAND ----------
# ============================================================
# CREATE SCORING STATE TABLE
# ============================================================

if new_transaction_count > 0:

    (
        current_max_processing_timestamp

        .withColumn(
            "updated_timestamp",
            F.current_timestamp()
        )

        .write

        .format("delta")

        .mode("overwrite")

        .option(
            "path",
            ML_SCORING_STATE_PATH
        )

        .saveAsTable(
            ML_SCORING_STATE_TABLE
        )
    )

    print(
        "Scoring state updated."
    )


# COMMAND ----------
# ============================================================
# FINAL VALIDATION
# ============================================================

if new_transaction_count > 0:

    display(

        spark.table(
            ML_PREDICTIONS_TABLE
        )

        .orderBy(
            F.col(
                "prediction_timestamp"
            )
            .desc()
        )

        .limit(20)
    )


# COMMAND ----------
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 70)

print(
    "AML ML INFERENCE COMPLETED"
)

print(
    "New transactions scored:",
    new_transaction_count
)

print(
    "Model:",
    MODEL_URI
)

print(
    "Feature count:",
    len(FEATURE_COLUMNS)
)

print(
    "ML Features Table:",
    ML_FEATURES_TABLE
)

print(
    "Prediction Table:",
    ML_PREDICTIONS_TABLE
)

print("=" * 70)